
1. What is the rate format?Normally, when you read data in Spark, you look at static files (like a CSV). The rate format is a special, built-in Spark tool used exclusively for testing. Instead of reading real files, it automatically generates a continuous stream of numbers and timestamps in memory out of thin air. You can control how fast it generates data (e.g., 5 rows per second). It acts as the "heartbeat" or engine driving your stream generator.2. What are the Columns?The script takes that raw "heartbeat" from the rate format and transforms it into realistic e-commerce data with these columns:event_id: A unique ID for every single click.customer_id: Random numbers representing which customer clicked.product_id: Random numbers representing which item they clicked on.event_type: A randomly chosen action. A user can either just look at an item (view), put it in their basket (add_to_cart), or buy it (purchase).timestamp: The exact date and second the click happened.

📂 landing_zone (Volume)
 ┣ 📂 batch/
 ┃ ┣ 📄 customers.csv
 ┃ ┗ 📄 products.csv
 ┗ 📂 stream/
   ┗ 📂 clickstream_json/
     ┣ 📄 part-00001.json  <-- (Appeared at 10:00:01 AM)
     ┣ 📄 part-00002.json  <-- (Appeared at 10:00:02 AM)
     ┗ 📄 part-00003.json  <-- (Appeared at 10:00:03 AM)


##Step 2.4: Generate Mock Streaming Data.

In [0]:
import pyspark.sql.functions as F

# ==========================================================
# Create Streaming Source
# ==========================================================
raw_stream = (
    spark.readStream
         .format("rate")
         .option("rowsPerSecond", 2)
         .load()
)

# ==========================================================
# Generate Mock Clickstream Data
# ==========================================================
clickstream_df = (
    raw_stream
    .withColumn("event_id", F.expr("uuid()"))

    # Customer IDs matching customers.csv
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("CUST_"),
            (F.floor(F.rand() * 100) + 1001).cast("int")
        )
    )

    # Product IDs matching products.csv
    .withColumn(
        "product_id",
        F.concat(
            F.lit("PROD_"),
            (F.floor(F.rand() * 25) + 2001).cast("int")
        )
    )

    # Event Types
    .withColumn(
        "event_type",
        F.element_at(
            F.array(
                F.lit("view"),
                F.lit("add_to_cart"),
                F.lit("purchase")
            ),
            F.floor(F.rand() * 3).cast("int") + 1
        )
    )

    .select(
        "event_id",
        "customer_id",
        "product_id",
        "event_type",
        F.col("timestamp")
    )
)

# ==========================================================
# Write Stream to JSON Files
# ==========================================================
query = (
    clickstream_df.writeStream
        .format("json")
        .option(
            "checkpointLocation",
            "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/checkpoints/generator"
        )
        .trigger(availableNow=True)
        .start(
            "/Volumes/dd_hr_sandpit/ecommerce_project/landing_zone/streaming/clickstream_json"
        )
)

query.awaitTermination()